# WORLDCRAFT — Colab GPU Worker
## PURE Pi3 + YOLO11-seg Reconstruction Server

**Setup steps:**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Copy the printed ngrok URL into `backend/.env`
4. Set `RECONSTRUCTION_MODE=colab` in `backend/.env`
5. Restart your local FastAPI server

⚠ Keep this notebook open. The session ends when you close it or hit the free-tier limit.

In [ ]:
# ── Cell 1: GPU check ──────────────────────────────────────────────────────
!nvidia-smi
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install packages (~3 min on fresh runtime) ────────────────────
!pip install -q transformers>=4.40 accelerate huggingface_hub
!pip install -q ultralytics
!pip install -q fastapi uvicorn python-multipart pyngrok pillow numpy
print('✓ Packages installed')

In [ ]:
# ── Cell 3: Load PURE Pi3 ─────────────────────────────────────────────────
from transformers import AutoModel, AutoImageProcessor
import torch

CHECKPOINT = 'FunnyGuy525/RefinedModel'
REVISION   = '45b24b6ec0963003281189455a2e04d517634133'

print(f'Loading PURE Pi3 from {CHECKPOINT}...')
processor  = AutoImageProcessor.from_pretrained(CHECKPOINT, revision=REVISION, trust_remote_code=True)
pure_model = AutoModel.from_pretrained(CHECKPOINT, revision=REVISION, trust_remote_code=True)
pure_model = pure_model.to(DEVICE).eval()
print('✓ PURE Pi3 loaded')

In [ ]:
# ── Cell 4: Load YOLO11-seg ───────────────────────────────────────────────
from ultralytics import YOLO
yolo_model = YOLO('yolo11n-seg.pt')  # ~6 MB, auto-downloaded
print('✓ YOLO11n-seg loaded')

In [ ]:
# ── Cell 5: Scene builder helpers ─────────────────────────────────────────
import numpy as np, math, uuid
from PIL import Image
from datetime import datetime, timezone

COCO_CLASSES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
    'traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat',
    'dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack',
    'umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball',
    'kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket',
    'bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple',
    'sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair',
    'couch','potted plant','bed','dining table','toilet','tv','laptop','mouse',
    'remote','keyboard','cell phone','microwave','oven','toaster','sink','refrigerator',
    'book','clock','vase','scissors','teddy bear','hair drier','toothbrush',
]

CLASS_TO_TYPE = {
    'couch':'sofa','chair':'chair','bed':'bed','dining table':'table',
    'tv':'storage','refrigerator':'storage','sink':'fixture',
    'toilet':'fixture','oven':'appliance','microwave':'appliance',
    'potted plant':'decor','vase':'decor','clock':'decor',
    'book':'decor','laptop':'decor',
}

OBJ_COLORS = ['#3d4452','#b58b57','#2b231d','#1a1d24','#d4af37',
               '#656b78','#4a7c59','#8b4513','#6b8cba','#c29b68']


def run_pure(images):
    inputs = processor.images(images)
    with torch.no_grad():
        return pure_model(**{k: v.to(DEVICE) for k, v in inputs.items()})


def run_yolo(images):
    return yolo_model.predict(images, verbose=False)


def build_scene(images, pure_out, yolo_results, scene_id):
    objects = []
    seen_types = {}

    for view_idx, (img, yolo_res) in enumerate(zip(images, yolo_results)):
        if yolo_res.masks is None:
            continue
        w, h = img.size
        try:
            pm = pure_out.points[view_idx].cpu().numpy()
            pm_h, pm_w = pm.shape[:2]
        except Exception:
            pm = None; pm_h = pm_w = None

        for det_idx in range(len(yolo_res.boxes)):
            cls_id   = int(yolo_res.boxes.cls[det_idx])
            conf     = float(yolo_res.boxes.conf[det_idx])
            cls_name = COCO_CLASSES[cls_id] if cls_id < len(COCO_CLASSES) else 'object'
            if conf < 0.4:
                continue
            obj_type = CLASS_TO_TYPE.get(cls_name, cls_name.replace(' ','_'))
            xyxy = yolo_res.boxes.xyxy[det_idx].cpu().numpy()
            bx, by, bx2, by2 = xyxy
            cx_px = (bx + bx2) / 2; cy_px = (by + by2) / 2
            box_w = (bx2 - bx) / w;  box_h = (by2 - by) / h

            if pm is not None:
                px = max(0, min(int(cx_px/w*pm_w), pm_w-1))
                py = max(0, min(int(cy_px/h*pm_h), pm_h-1))
                x3, y3, z3 = pm[py, px].tolist()
            else:
                x3 = (cx_px/w - 0.5)*5.0; y3 = 0.5; z3 = (cy_px/h - 0.5)*3.0

            est_w = max(box_w*5.0, 0.2)
            est_h = max(box_h*2.8, 0.2)
            est_d = max(box_w*5.0, 0.2)

            obj = {
                'id': f'obj-{cls_name.replace(" ","_")}-{view_idx:02d}-{det_idx:02d}',
                'type': obj_type, 'name': cls_name.replace('_',' ').title(),
                'position': {'x':round(x3,3),'y':round(est_h/2,3),'z':round(z3,3)},
                'dimensions': {'width':round(est_w,2),'height':round(est_h,2),'depth':round(est_d,2)},
                'rotation': {'x':0,'y':0,'z':0}, 'movable':True,
                'confidence': round(conf,3),
                'color': OBJ_COLORS[len(objects) % len(OBJ_COLORS)],
                'material': 'Detected surface', 'category': obj_type,
                'price': None, 'selected': False,
                'provenance': 'yolo11',
                'uncertainty_note': 'Coordinates model-relative; metric scale unverified.',
            }
            # Keep highest confidence per type
            if obj_type not in seen_types or conf > seen_types[obj_type]['confidence']:
                seen_types[obj_type] = obj

    objects = list(seen_types.values())

    return {
        'scene_id': scene_id, 'name': 'Reconstructed Room (PURE Pi3 + YOLO11-seg)',
        'schema_version': 1, 'units': 'meters',
        'coordinate_system': {'up_axis':'Y','description':'Application-defined right-handed'},
        'room': {
            'type': 'living_room',
            'dimensions': {'width':5.4,'length':6.2,'height':2.8,'status':'estimated_pure_pi3'},
            'wall_color':'#202632','accent_wall_color':'#1d232e','floor_color':'#36302a',
        },
        'objects': objects,
        'relationships': [], 'constraints': [], 'version': 1,
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'prompt_history': [f'Reconstructed from {len(images)} image(s) via PURE Pi3 + YOLO11-seg'],
        'input_type': 'image', 'reconstruction_model': 'pure_pi3_colab',
        'validation': {
            'collisions':0,'clearance_ok':True,'pathway_clear':True,'boundary_ok':True,
            'violations':[],
            'messages':[
                f'PURE Pi3: {len(images)} view(s) processed',
                f'YOLO11-seg: {len(objects)} unique objects identified',
                'NOTE: Coordinates are model-relative — metric scale unverified.',
                'NOTE: Cross-view alignment is approximate.',
            ],
        },
    }

print('✓ Helpers ready')

In [ ]:
# ── Cell 6: FastAPI worker server ─────────────────────────────────────────
import io, traceback
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn, threading

worker = FastAPI(title='WORLDCRAFT Colab Worker')
worker.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])


@worker.get('/health')
def health():
    return {'status':'ok','device':DEVICE,'pure_loaded':True,'yolo_loaded':True}


@worker.post('/reconstruct')
async def reconstruct(
    files: list[UploadFile] = File(default=[]),
    scene_id: str = Form(default=''),
):
    try:
        if not files:
            return JSONResponse({'error': 'No files provided'}, status_code=400)
        images = [Image.open(io.BytesIO(await f.read())).convert('RGB') for f in files]
        sid = scene_id or f'scene-{uuid.uuid4().hex[:8]}'
        print(f'[WORLDCRAFT] Reconstructing {len(images)} image(s) — scene {sid}')
        pure_out  = run_pure(images)
        yolo_res  = run_yolo(images)
        scene     = build_scene(images, pure_out, yolo_res, sid)
        print(f'[WORLDCRAFT] Done — {len(scene["objects"])} object(s) detected')
        return {'scene_id': sid, 'status': 'reconstructed', 'scene': scene}
    except Exception:
        tb = traceback.format_exc()
        print(tb)
        return JSONResponse({'error': tb[-600:]}, status_code=500)


thread = threading.Thread(
    target=lambda: uvicorn.run(worker, host='0.0.0.0', port=8001, log_level='warning'),
    daemon=True,
)
thread.start()
import time; time.sleep(2)
print('✓ FastAPI worker running on port 8001')

In [ ]:
# ── Cell 7: Expose via ngrok ───────────────────────────────────────────────
from pyngrok import ngrok

# ⚠ Replace with YOUR ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = 'YOUR_NGROK_AUTH_TOKEN_HERE'

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8001).public_url

print('=' * 62)
print('✓  WORLDCRAFT Colab GPU worker is LIVE')
print(f'   Public URL : {public_url}')
print()
print('   Paste into  backend/.env :')
print(f'     COLAB_ENDPOINT_URL={public_url}')
print('     RECONSTRUCTION_MODE=colab')
print()
print('   Then restart the FastAPI server:')
print('     cd backend')
print('     uvicorn main:app --reload --port 8000')
print('=' * 62)
print()
print('⚠  Keep this notebook open. The URL changes each session.')

In [ ]:
# ── Cell 8 (optional): Quick health-check ────────────────────────────────
import requests
r = requests.get(f'{public_url}/health', headers={'ngrok-skip-browser-warning':'true'})
print('Health check:', r.json())